In [0]:
!pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [01:00<00:00, 75.8MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-0454bcc6-7bd4-427e-80a0-3c nogroup 8.4G Jan 12 19:13 2019-Nov.csv
-rwxrwxrwx 1 spark-0454bcc6-7bd4-427e-80a0-3c nogroup 5.3G Jan 12 19:15 2019-Oct.csv
-rwxrwxrwx 1 spark-0454bcc6-7bd4-427e-80a0-3c nogroup 4.3G Jan 12 19:12 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 12 18:34 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-0454bcc6-7bd4-427e-80a0-3c nogroup 8.4G Jan 12 19:13 2019-Nov.csv
-rwxrwxrwx 1 spark-0454bcc6-7bd4-427e-80a0-3c nogroup 5.3G Jan 12 19:15 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 12 18:34 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
input_csv = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"

df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(input_csv)
)
df0 = (df
       .withColumnRenamed("event_time", "event_time")
       .withColumn("event_ts", F.to_timestamp("event_time"))   # parses '2019-10-01 00:00:00 UTC' style
       .withColumn("event_date", F.to_date("event_ts"))
       .withColumn("event_hour", F.date_trunc("hour", F.col("event_ts")))  # trunc to hour [web:70]
       .withColumn("price", F.col("price").cast("double"))
      )

df0.select("event_ts","event_type","product_id","category_code","brand","price","user_id","user_session").show(5, truncate=False)

display(df.limit(10))
df.printSchema()


+-------------------+----------+----------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_ts           |event_type|product_id|category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:00:01|view      |1307067   |computers.notebook                 |lenovo  |251.74 |550050854|7c90fc70-0e80-4590-96f3-13c02c

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-01T00:00:00.000Z,view,44600062,2103807459595387724,null,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
2019-10-01T00:00:00.000Z,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.2,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2019-10-01T00:00:01.000Z,view,17200506,2053013559792632471,furniture.living_room.sofa,null,543.1,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
2019-10-01T00:00:01.000Z,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
2019-10-01T00:00:04.000Z,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
2019-10-01T00:00:05.000Z,view,1480613,2053013561092866779,computers.desktop,pulser,908.62,512742880,0d0d91c2-c9c2-4e81-90a5-86594dec0db9
2019-10-01T00:00:08.000Z,view,17300353,2053013553853497655,null,creed,380.96,555447699,4fe811e9-91de-46da-90c3-bbd87ed3a65d
2019-10-01T00:00:08.000Z,view,31500053,2053013558031024687,null,luminarc,41.16,550978835,6280d577-25c8-4147-99a7-abc6048498d6
2019-10-01T00:00:10.000Z,view,28719074,2053013565480109009,apparel.shoes.keds,baden,102.71,520571932,ac1cd4e5-a3ce-4224-a2d7-ff660a105880
2019-10-01T00:00:11.000Z,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [0]:
session_agg = (
    df0.groupBy("user_session")
       .agg(
           F.min("event_ts").alias("session_start_ts"),
           F.max("event_ts").alias("session_end_ts"),
           F.count("*").alias("session_events"),
           F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("session_purchases"),
           F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("session_revenue")
       )
)
events_with_session = (
    df0.join(session_agg, on="user_session", how="left")
       .withColumn("session_duration_sec",
                   (F.col("session_end_ts").cast("long") - F.col("session_start_ts").cast("long")))
)

display(events_with_session.select("user_session","event_ts","event_type","price","session_events","session_purchases","session_revenue","session_duration_sec").limit(20))

user_session,event_ts,event_type,price,session_events,session_purchases,session_revenue,session_duration_sec
af23c635-2105-4d65-a538-7642a026f07a,2019-10-02T14:26:49.000Z,view,77.2,6,0,0.0,311
d6089885-5f4c-4b11-9cff-aa4f3f768e7d,2019-10-02T14:26:49.000Z,view,23.14,7,0,0.0,1607
ab8c69f7-5e5c-45e5-9d94-301761efe485,2019-10-02T14:26:49.000Z,view,218.77,7,0,0.0,525
26db4f41-880d-42e9-b443-38bc8fa93008,2019-10-04T08:50:53.000Z,view,292.08,4,0,0.0,472
adfbaea3-4d07-46f7-b47b-888b7215146c,2019-10-05T16:36:18.000Z,cart,252.55,21,1,252.55,1006
4535326e-4c24-4f86-8fa9-b2e04e5e2d3a,2019-10-05T16:36:18.000Z,view,179.93,10,0,0.0,681
b6401e78-7207-4846-b8c9-9ba0314a3db0,2019-10-04T08:50:53.000Z,view,223.68,2,0,0.0,439
a0f8f27e-19d4-434f-81fa-5f7487c42fad,2019-10-07T08:46:35.000Z,view,244.54,8,0,0.0,216
c6a9300c-6ffd-42a1-bce8-178ded5523da,2019-10-05T16:36:19.000Z,view,65.36,56,0,0.0,1418
fe73f3ea-858a-43e3-8776-dbee8483e967,2019-10-05T16:36:19.000Z,view,250.74,5,0,0.0,458


In [0]:
w_seq = Window.partitionBy("user_session").orderBy("event_ts")

seq = (
    df0.select(
        "user_session","event_ts","event_type","product_id","price",
        F.row_number().over(w_seq).alias("rn")
    )
)

seq_joined = (
    seq.alias("a")
       .join(
           seq.alias("b"),
           on=[
               F.col("a.user_session") == F.col("b.user_session"),
               F.col("a.rn") + F.lit(1) == F.col("b.rn")
           ],
           how="left"
       )
       .select(
           F.col("a.user_session").alias("user_session"),
           F.col("a.event_ts").alias("event_ts"),
           F.col("a.event_type").alias("event_type"),
           F.col("b.event_ts").alias("next_event_ts"),
           F.col("b.event_type").alias("next_event_type"),
           (F.col("b.event_ts").cast("long") - F.col("a.event_ts").cast("long")).alias("secs_to_next_event")
       )
)

display(seq_joined.limit(20))

user_session,event_ts,event_type,next_event_ts,next_event_type,secs_to_next_event
00002e45-650a-4dbb-bbe6-29d709fb652f,2019-10-13T08:48:01.000Z,view,2019-10-13T08:49:48.000Z,view,107
00004ada-8f93-49a6-956d-4ed71ae94791,2019-10-17T13:36:05.000Z,view,2019-10-17T13:36:52.000Z,view,47
00000809-9101-4e4b-9795-e6cbafccfe19,2019-10-25T16:03:44.000Z,view,2019-10-25T16:05:18.000Z,view,94
00000083-8816-4d58-a9b8-f52f54186edc,2019-10-06T11:30:18.000Z,view,2019-10-06T11:30:25.000Z,view,7
000024a4-d991-4020-a163-f6fdcc80efac,2019-10-07T19:24:42.000Z,view,2019-10-07T19:28:06.000Z,view,204
0000c61b-4cee-466a-97e8-cdd7c2ff6562,2019-10-16T06:58:59.000Z,view,2019-10-16T06:59:03.000Z,view,4
00000056-a206-40dd-b174-a072550fa38c,2019-10-31T06:31:25.000Z,view,null,null,null
00003599-a772-4c8a-9c22-0dfa4f6ecc83,2019-10-09T21:39:14.000Z,view,2019-10-09T21:39:19.000Z,view,5
00003599-a772-4c8a-9c22-0dfa4f6ecc83,2019-10-09T21:41:23.000Z,view,2019-10-09T21:41:39.000Z,view,16
00004ada-8f93-49a6-956d-4ed71ae94791,2019-10-17T13:35:24.000Z,purchase,2019-10-17T13:36:05.000Z,view,41


In [0]:
purchase_events = (
    df0.filter(F.col("event_type") == "purchase")
       .select("user_id","user_session","event_ts","price","product_id","brand","category_code")
)

w_user_running = (
    Window.partitionBy("user_id")
          .orderBy("event_ts")
          .rowsBetween(Window.unboundedPreceding, 0)
)

w_session_running = (
    Window.partitionBy("user_session")
          .orderBy("event_ts")
          .rowsBetween(Window.unboundedPreceding, 0)
)

purchase_with_running = (
    purchase_events
      .withColumn("running_revenue_user", F.sum("price").over(w_user_running))
      .withColumn("running_revenue_session", F.sum("price").over(w_session_running))
)

display(purchase_with_running.orderBy("user_id","event_ts").limit(50))

user_id,user_session,event_ts,price,product_id,brand,category_code,running_revenue_user,running_revenue_session
264649825,b128149a-c44b-46f8-9029-780546d8e398,2019-10-06T15:29:00.000Z,552.14,8500083,kiturami,null,552.14,552.14
264649825,b1265f21-cf39-49bd-af0a-88a1e2ef960a,2019-10-06T20:36:11.000Z,687.9,8500084,kiturami,null,1240.04,687.9
303160429,8957377b-66b3-4661-ae91-5c2b5e29fd0e,2019-10-13T13:25:29.000Z,340.59,5100443,garmin,electronics.clocks,340.59,340.59
340041246,c0c5b9c2-29bb-43ff-98f0-6a53abd70d50,2019-10-03T11:59:28.000Z,200.52,9300037,lg,null,200.52,200.52
340041246,c0c5b9c2-29bb-43ff-98f0-6a53abd70d50,2019-10-05T05:48:46.000Z,174.78,9300037,lg,null,375.3,375.3
340041246,a33db1e9-d71d-4a0e-b969-92664aae3170,2019-10-11T04:49:29.000Z,200.52,9300037,lg,null,575.82,200.52
340041246,596ea40e-80c7-4c30-bca4-60513fec1925,2019-10-14T14:45:39.000Z,339.7,2100099,lg,electronics.video.tv,915.52,339.7
371877401,d3103132-d38a-4399-9404-f99d03d243d0,2019-10-09T15:51:08.000Z,29.89,17301479,null,null,29.89,29.89
384989212,7472f245-e885-4c5e-b3b1-476a7b19b508,2019-10-18T11:19:59.000Z,41.44,2501816,artel,appliances.kitchen.oven,41.44,41.44
387300134,51d42df3-8ba8-493f-b780-41f09a74be28,2019-10-26T14:05:39.000Z,20.59,10700971,null,null,20.59,20.59


In [0]:
feat = (events_with_session
        .withColumn("hour_of_day", F.hour("event_ts"))
        .withColumn("day_of_week", F.dayofweek("event_ts"))  # 1=Sun ... 7=Sat
        .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]))  # Sun or Sat
       )

feat = (feat
        .withColumn("is_view", F.col("event_type") == F.lit("view"))
        .withColumn("is_cart", F.col("event_type") == F.lit("cart"))
        .withColumn("is_purchase", F.col("event_type") == F.lit("purchase"))
       )

feat = (feat
        .withColumn("category_l1", F.split(F.col("category_code"), "\\.").getItem(0))
        .withColumn("category_l2", F.split(F.col("category_code"), "\\.").getItem(1))
       )

feat = (feat
        .withColumn("session_converted", F.col("session_purchases") > F.lit(0))
        .withColumn("avg_purchase_price_in_session",
                    F.when(F.col("session_purchases") > 0, F.col("session_revenue") / F.col("session_purchases")))
       )

display(feat.select(
    "event_ts","event_type","price","brand","category_code","category_l1","category_l2",
    "session_events","session_purchases","session_revenue","session_converted","avg_purchase_price_in_session"
).limit(30))

event_ts,event_type,price,brand,category_code,category_l1,category_l2,session_events,session_purchases,session_revenue,session_converted,avg_purchase_price_in_session
2019-10-02T14:26:49.000Z,view,77.2,klima,appliances.environment.water_heater,appliances,environment,6,0,0.0,false,null
2019-10-02T14:26:49.000Z,view,23.14,null,auto.accessories.player,auto,accessories,7,0,0.0,false,null
2019-10-01T16:59:08.000Z,view,280.55,indesit,appliances.kitchen.refrigerators,appliances,kitchen,55,0,0.0,false,null
2019-10-02T14:26:49.000Z,view,218.77,null,null,null,null,7,0,0.0,false,null
2019-10-01T00:00:24.000Z,view,151.87,null,null,null,null,2,0,0.0,false,null
2019-10-02T14:26:49.000Z,view,823.44,samsung,appliances.kitchen.refrigerators,appliances,kitchen,10,0,0.0,false,null
2019-10-01T00:00:25.000Z,view,122.18,ariston,appliances.environment.water_heater,appliances,environment,2,0,0.0,false,null
2019-10-01T00:00:23.000Z,view,47.62,midea,appliances.environment.air_heater,appliances,environment,1,0,0.0,false,null
2019-10-03T12:26:51.000Z,view,54.03,microsoft,null,null,null,2,0,0.0,false,null
2019-10-03T12:26:50.000Z,view,1377.25,firman,construction.tools.generator,construction,tools,61,0,0.0,false,null


In [0]:
feat_dir = "/Volumes/workspace/ecommerce/ecommerce_data/outputs/feat_csv"
(feat
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(feat_dir)
)
print("CSV written to:", feat_dir)

CSV written to: /Volumes/workspace/ecommerce/ecommerce_data/outputs/feat_csv


In [0]:
run_dir = "/Volumes/workspace/ecommerce/ecommerce_data/outputs/run_csv"

(purchase_with_running
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(run_dir)
)

print("CSV written to:", run_dir)


CSV written to: /Volumes/workspace/ecommerce/ecommerce_data/outputs/run_csv


In [0]:
(feat.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce.feat")
)

(purchase_with_running.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce.run")
)

(seq_joined.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce.session_next_event")
)

display(spark.table("workspace.ecommerce.feat").limit(10))


user_session,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,event_ts,event_date,event_hour,session_start_ts,session_end_ts,session_events,session_purchases,session_revenue,session_duration_sec,hour_of_day,day_of_week,is_weekend,is_view,is_cart,is_purchase,category_l1,category_l2,session_converted,avg_purchase_price_in_session
159b4a84-1687-48a9-a718-10eb8c612bc8,2019-10-01T02:20:45.000Z,view,16400029,2053013558249128509,null,tefal,61.75,513070042,2019-10-01T02:20:45.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:20:45.000Z,2019-10-01T02:22:24.000Z,4,1,38.59,99,2,3,false,true,false,false,null,null,true,38.59
159b4a84-1687-48a9-a718-10eb8c612bc8,2019-10-01T02:21:23.000Z,view,15900073,2053013558190408249,null,tefal,38.59,513070042,2019-10-01T02:21:23.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:20:45.000Z,2019-10-01T02:22:24.000Z,4,1,38.59,99,2,3,false,true,false,false,null,null,true,38.59
a1a728b6-1ece-4631-bc1b-5d6a4d74db7e,2019-10-01T02:21:51.000Z,view,1004874,2053013555631882655,electronics.smartphone,samsung,383.51,513822280,2019-10-01T02:21:51.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:21:51.000Z,2019-10-01T02:22:11.000Z,3,0,0.0,20,2,3,false,true,false,false,electronics,smartphone,false,null
159b4a84-1687-48a9-a718-10eb8c612bc8,2019-10-01T02:21:53.000Z,purchase,15900073,2053013558190408249,null,tefal,38.59,513070042,2019-10-01T02:21:53.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:20:45.000Z,2019-10-01T02:22:24.000Z,4,1,38.59,99,2,3,false,false,false,true,null,null,true,38.59
a1a728b6-1ece-4631-bc1b-5d6a4d74db7e,2019-10-01T02:22:05.000Z,view,1004874,2053013555631882655,electronics.smartphone,samsung,383.51,513822280,2019-10-01T02:22:05.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:21:51.000Z,2019-10-01T02:22:11.000Z,3,0,0.0,20,2,3,false,true,false,false,electronics,smartphone,false,null
a1a728b6-1ece-4631-bc1b-5d6a4d74db7e,2019-10-01T02:22:11.000Z,view,1004873,2053013555631882655,electronics.smartphone,samsung,388.81,513822280,2019-10-01T02:22:11.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:21:51.000Z,2019-10-01T02:22:11.000Z,3,0,0.0,20,2,3,false,true,false,false,electronics,smartphone,false,null
159b4a84-1687-48a9-a718-10eb8c612bc8,2019-10-01T02:22:24.000Z,view,15900073,2053013558190408249,null,tefal,38.59,513070042,2019-10-01T02:22:24.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:20:45.000Z,2019-10-01T02:22:24.000Z,4,1,38.59,99,2,3,false,true,false,false,null,null,true,38.59
a29c2788-449b-41c3-82c6-f87ea45914c5,2019-10-01T02:23:57.000Z,view,44700012,2104564977229628393,null,lux,52.1,513812981,2019-10-01T02:23:57.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:23:57.000Z,2019-10-01T02:26:31.000Z,3,0,0.0,154,2,3,false,true,false,false,null,null,false,null
a29c2788-449b-41c3-82c6-f87ea45914c5,2019-10-01T02:26:07.000Z,view,12718243,2053013553559896355,null,comforser,37.32,513812981,2019-10-01T02:26:07.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:23:57.000Z,2019-10-01T02:26:31.000Z,3,0,0.0,154,2,3,false,true,false,false,null,null,false,null
a29c2788-449b-41c3-82c6-f87ea45914c5,2019-10-01T02:26:31.000Z,view,12704241,2053013553559896355,null,triangle,41.19,513812981,2019-10-01T02:26:31.000Z,2019-10-01,2019-10-01T02:00:00.000Z,2019-10-01T02:23:57.000Z,2019-10-01T02:26:31.000Z,3,0,0.0,154,2,3,false,true,false,false,null,null,false,null
